# OushudhCheck: Fine-Tune `google/gemma-3-1b-it` on Kaggle

Trains and exports a fast, RAG-grounded **Bangla Medicine Safety SLM** (`unsloth/gemma-3-1b-it-unsloth-bnb-4bit`) directly from `medicines.db`.

### Output Style
- Keeps **medicine brand name, numeric strength (`mg`/`ml`), and generic name in exact English** so there are zero transliteration typos or dose hallucinations (e.g., `Seclo (20 mg) (ক্যাপসুল) ওষুধটির মূল জেনেরিক উপাদান হলো Omeprazole। ...`).
- Writes **all explanations, dosage forms, meal timing, side effects, and safety warnings in simple, clear Bangla**.

In [15]:
!pip install -q unsloth trl datasets accelerate bitsandbytes

In [16]:
from pathlib import Path
import torch

gpu_count = torch.cuda.device_count()
assert gpu_count >= 1, "No GPU found!"

for idx in range(gpu_count):
    props = torch.cuda.get_device_properties(idx)
    print(f"GPU {idx}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

OUTPUT_DIR = Path("/kaggle/working/oushudhcheck_gemma3_1b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

GPU 0: Tesla T4 | VRAM: 15.6 GB
GPU 1: Tesla T4 | VRAM: 15.6 GB
Output directory: /kaggle/working/oushudhcheck_gemma3_1b


## Load Directly from `medicines.db`
Reads the `medicines` SQLite view directly from `/kaggle/input/**/medicines.db`.

In [17]:
import sqlite3
import pandas as pd

DB_PATH = next(Path("/kaggle/input").rglob("medicines.db"))
print(f"Loading SQLite database: {DB_PATH}")

conn = sqlite3.connect(str(DB_PATH))
meds_df = pd.read_sql_query(
    """
    SELECT
        brand_name,
        type,
        dosage_form,
        generic,
        generic_lower,
        manufacturer,
        package_container,
        drug_class,
        indication,
        dosage_description,
        side_effects_description,
        interaction_description,
        contraindications_description,
        precautions_description,
        pregnancy_and_lactation_description,
        beginner_summary
    FROM medicines
    WHERE brand_name IS NOT NULL
      AND generic IS NOT NULL
    """,
    conn,
)
conn.close()

meds_df = meds_df.fillna("")
print(f"Loaded {len(meds_df):,} records across {meds_df['generic_lower'].nunique():,} unique generics.")
meds_df[["brand_name", "generic", "dosage_form", "package_container"]].head(5)

Loading SQLite database: /kaggle/input/datasets/ridoyhossan/bd-medicine-datasheet/medicines.db
Loaded 17,191 records across 1,661 unique generics.


,brand_name,generic,dosage_form,package_container
0,A-Cold,Bromhexine Hydrochloride,Syrup,100 ml bottle
1,A-Cof,Dextromethorphan + Pseudoephedrine + Triprolidine,Syrup,100 ml bottle
2,A-Clox,Cloxacillin Sodium,Injection,250 mg vial
3,A-Clox,Cloxacillin Sodium,Powder for Suspension,100 ml bottle
4,A-Clox,Cloxacillin Sodium,Capsule,


## Build Grounded Bangla SFT Dataset (Exact English Brand / Strength / Generic + Bangla Explanation)
- Keeps `brand_name`, numeric `strength` (`500 mg + 65 mg`, `20 mg`), and `generic` in **exact English** so numbers and drug names are never corrupted or hallucinated.
- Writes all medical explanations, dosage forms (`ট্যাবলেট`, `ক্যাপসুল`, `সিরাপ`), meal timings, side effects, and precautions in **clean, grammatical Bangla**.
- Handles negation (`no side effects`, `well tolerated`) and separates medical conditions from action warnings (`অ্যালকোহল এড়িয়ে চলুন`, `গাড়ি চালানো থেকে বিরত থাকুন`).
- Samples up to 2 brands per unique `generic_lower` for balanced coverage across all 1,661 generics. (If you upload a teacher-distilled `custom_bangla_sft.jsonl` to `/kaggle/input` or `OUTPUT_DIR`, it will be used automatically.)

In [18]:
import json
import random
import re
import textwrap

SYSTEM_PROMPT = (
    "তুমি OushudhCheck মেডিকেল সহায়ক। শুধুমাত্র প্রদত্ত ডাটাবেস তথ্যের ভিত্তিতে "
    "রোগীর জন্য ৩-৪ বাক্যে সহজ ও স্পষ্ট বাংলায় ওষুধটির কাজ, খাওয়ার নিয়ম ও সময় এবং "
    "গুরুত্বপূর্ণ সতর্কতা লেখো। ওষুধের ব্র্যান্ড নাম, মাত্রা (যেমন 20 mg বা 500 mg + 65 mg) "
    "এবং জেনেরিক উপাদানের নাম ডাটাবেসে যেভাবে ইংরেজিতে আছে ঠিক সেভাবেই অপরিবর্তিত রাখবে, "
    "আর বাকি সম্পূর্ণ ব্যাখ্যা সহজ বাংলায় লিখবে। ডাটাবেসে নেই এমন কোনো মাত্রা, পার্শ্বপ্রতিক্রিয়া বা "
    "সতর্কতা মনগড়াভাবে যোগ করবে না।"
)

(OUTPUT_DIR / "system_prompt.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")

PATIENT_QUESTIONS = [
    "এই ওষুধটি কী কাজ করে, কখন খাবো এবং কী সতর্কতা মানতে হবে?",
    "সহজ বাংলায় বলুন এই ওষুধের কাজ কী, খাওয়ার নিয়ম কী এবং কী কী সাবধানতা দরকার?",
    "এই ওষুধটা কি খাবারের আগে না পরে খেতে হয়? এর কাজ ও পার্শ্বপ্রতিক্রিয়া কী?",
    "রোগীর জন্য সহজ বাংলায় এই ওষুধটির ৩-৪ বাক্যের সারসংক্ষেপ দিন।",
    "বয়স্ক রোগীর জন্য এই ওষুধ খাওয়ার নিয়ম, কাজ ও জরুরি সতর্কতা সহজ বাংলায় লিখুন।",
]

FORM_BN = {
    "tablet": "ট্যাবলেট",
    "chewable tablet": "চিবিয়ে খাওয়ার ট্যাবলেট",
    "dispersible tablet": "পানিতে গুলিয়ে খাওয়ার ট্যাবলেট",
    "tablet (extended release)": "ধীরে কার্যকর ট্যাবলেট",
    "tablet (modified release)": "নিয়ন্ত্রিত মাত্রার ট্যাবলেট",
    "tablet (sustained release)": "দীর্ঘস্থায়ী ট্যাবলেট",
    "capsule": "ক্যাপসুল",
    "capsule (extended release)": "ধীরে কার্যকর ক্যাপসুল",
    "capsule (sustained release)": "দীর্ঘস্থায়ী ক্যাপসুল",
    "capsule (timed release)": "নিয়ন্ত্রিত মাত্রার ক্যাপসুল",
    "syrup": "সিরাপ",
    "oral suspension": "ওরাল সাসপেনশন",
    "suspension": "সাসপেনশন",
    "powder for suspension": "সাসপেনশন তৈরির পাউডার",
    "oral solution": "তরল ওষুধ",
    "liquid": "তরল ওষুধ",
    "pediatric drops": "শিশুদের ড্রপ",
    "drop": "ড্রপ",
    "injection": "ইনজেকশন",
    "im/iv injection": "ইনজেকশন",
    "iv injection": "শিরায় দেওয়ার ইনজেকশন",
    "im injection": "পেশিতে দেওয়ার ইনজেকশন",
    "sc injection": "ত্বকের নিচে দেওয়ার ইনজেকশন",
    "iv infusion": "স্যালাইন বা শিরায় ইনফিউশন",
    "iv injection or infusion": "ইনজেকশন বা ইনফিউশন",
    "im/ia injection": "ইনজেকশন",
    "cream": "ক্রিম",
    "ointment": "মলম",
    "gel": "জেল",
    "topical gel": "বাহ্যিক ব্যবহারের জেল",
    "lotion": "লোশন",
    "muscle rub": "ব্যথার মলম",
    "scalp solution": "মাথার ত্বকের লোশন",
    "ophthalmic solution": "চোখের ড্রপ",
    "ophthalmic suspension": "চোখের ড্রপ",
    "ophthalmic ointment": "চোখের মলম",
    "nasal spray": "নাকের স্প্রে",
    "nasal drop": "নাকের ড্রপ",
    "inhaler": "ইনহেলার",
    "nebuliser solution": "নেবুলাইজার সলিউশন",
    "suppository": "সাপোজিটরি",
    "vaginal tablet": "যোনিপথে ব্যবহারের ট্যাবলেট",
    "mouthwash": "মাউথওয়াশ",
    "effervescent powder": "পানিতে গুলিয়ে খাওয়ার পাউডার",
    "effervescent granules": "পানিতে গুলিয়ে খাওয়ার দানা",
    "oral powder": "খাওয়ার পাউডার",
}

INDICATION_BN_RULES = [
    (r"tuberculosis|\btb\b", "যক্ষ্মা (Tuberculosis)"),
    (r"malaria", "ম্যালেরিয়া"),
    (r"fever|pyrexia|influenza", "জ্বর"),
    (r"migraine", "মাইগ্রেন"),
    (r"headache", "মাথাব্যথা"),
    (r"toothache|dental", "দাঁতব্যথা"),
    (r"osteoarthritis|rheumatoid|arthritis|gout|spondylitis", "বাতের ব্যথা ও প্রদাহ"),
    (r"back pain|muscle pain|spasm|sprain|pain|analgesic", "শরীর ও মাংসপেশির ব্যথা"),
    (r"peptic ulcer|gastric ulcer|duodenal ulcer|ulcer", "পেপটিক আলসার"),
    (r"gerd|reflux|heartburn|acidity|excess stomach acid|gastritis", "গ্যাস্ট্রিক, অ্যাসিডিটি ও বুকজ্বালা"),
    (r"indigestion|dyspepsia|flatulence|bloating", "বদহজম ও পেট ফাঁপা"),
    (r"constipation|laxative|piles|hemorrhoid", "কোষ্ঠকাঠিন্য"),
    (r"diarrh|dysentery|giardiasis|amoebic|ibs", "ডায়রিয়া ও পেটের পীড়া"),
    (r"nausea|vomiting", "বমি ভাব ও বমি"),
    (r"hypertension|high blood pressure", "উচ্চ রক্তচাপ"),
    (r"angina|heart failure|arrhythmia|ischaemic|cardiac", "হৃদরোগ ও বুকে ব্যথা"),
    (r"stroke|thrombo|blood clot|embolism", "রক্ত জমাট বাঁধা ও স্ট্রোক প্রতিরোধ"),
    (r"cholesterol|hypercholesterol|lipid", "রক্তের অতিরিক্ত কোলেস্টেরল"),
    (r"diabetes|type 2 dm|blood sugar|glycaemic", "ডায়াবেটিস ও রক্তের শর্করা"),
    (r"thyroid|hypothyroid|hyperthyroid", "থাইরয়েডের সমস্যা"),
    (r"asthma|copd|bronchitis|wheezing", "হাঁপানি ও শ্বাসকষ্ট"),
    (r"cough|cold|phlegm|catarrh|rhinitis|sinusitis", "সর্দি-কাশি ও কফ"),
    (r"allergy|allergic|hives|urticaria|itching|sneezing|runny nose", "অ্যালার্জি ও চুলকানি"),
    (r"urinary tract|uti|cystitis", "প্রস্রাবের সংক্রমণ"),
    (r"pneumonia|tonsillitis|pharyngitis|typhoid|bacterial|infection", "ব্যাকটেরিয়াজনিত সংক্রমণ"),
    (r"fungal|candidiasis|ring worm|dermatomycos", "ছত্রাকজনিত সংক্রমণ"),
    (r"viral|herpes|hepatitis|hiv|chickenpox", "ভাইরাসজনিত সংক্রমণ"),
    (r"worm|helminth", "কৃমি সংক্রমণ"),
    (r"anxiety|panic|restlessness|tension", "অতিরিক্ত উদ্বেগ ও অস্থিরতা"),
    (r"insomnia|trouble sleeping|sleep", "অনিদ্রা"),
    (r"depression", "মানসিক অবসাদ"),
    (r"seizure|epilepsy|convulsion", "খিঁচুনি"),
    (r"vertigo|dizziness|vestibular", "মাথা ঘোরা (Vertigo)"),
    (r"neuropath|nerve pain", "স্নায়ুর ব্যথা"),
    (r"acne|eczema|psoriasis|scabies|dermatitis|skin|wound|burn", "ত্বকের সমস্যা ও প্রদাহ"),
    (r"eye|ocular|conjunctiv|cataract|glaucoma", "চোখের অ্যালার্জি ও প্রদাহ"),
    (r"ear|otitis", "কানের সংক্রমণ ও ব্যথা"),
    (r"vitamin|deficiency|supplement|malnutrition|weakness|fatigue", "ভিটামিন ঘাটতি ও শারীরিক দুর্বলতা"),
    (r"anemia|anaemia|iron deficiency", "রক্তস্বল্পতা"),
    (r"osteoporosis|rickets|bone|calcium", "হাড়ের দুর্বলতা ও ক্যালসিয়াম ঘাটতি"),
    (r"jaundice|liver|hepatic", "লিভারের সুরক্ষা ও জন্ডিস"),
    (r"prostate|bph|urinary", "প্রোস্টেট ও প্রস্রাবের সমস্যা"),
    (r"menstrual|dysmenorrhea|leucorrhoea|menopausal|pcos", "নারীস্বাস্থ্য ও মাসিকজনিত সমস্যা"),
    (r"cancer|carcinoma|tumor|leukemia|chemotherapy", "ক্যান্সারের চিকিৎসা"),
]

CLASS_BN_FALLBACK = [
    (r"antibiotic|penicillin|cephalosporin|macrolide|quinolone|aminoglycoside", "ব্যাকটেরিয়াজনিত সংক্রমণ নিরাময়ে"),
    (r"proton pump|ppi|h2|antacid", "গ্যাস্ট্রিক ও অ্যাসিডিটি নিয়ন্ত্রণে"),
    (r"nsaid|analgesic|anti-inflammatory", "ব্যথা, জ্বর ও প্রদাহ কমাতে"),
    (r"anti-histamine|antihistamine|allergic", "অ্যালার্জি, হাঁচি ও চুলকানি উপশমে"),
    (r"hypertens|beta blocker|calcium channel|arb|ace inhibitor|nitrate", "উচ্চ রক্তচাপ ও হৃদরোগ নিয়ন্ত্রণে"),
    (r"diabetic|hypoglyc|insulin|sulfonylurea|dpp|sglt", "ডায়াবেটিস ও রক্তের শর্করা নিয়ন্ত্রণে"),
    (r"bronchodilator|asthma|respiratory", "শ্বাসকষ্ট ও হাঁপানি নিয়ন্ত্রণে"),
    (r"benzodiazepine|anxiolytic|sedative|antidepressant", "উদ্বেগ, অনিদ্রা ও স্নায়বিক চাপ কমাতে"),
    (r"antifungal", "ছত্রাকজনিত সংক্রমণ দূর করতে"),
    (r"antiviral", "ভাইরাসজনিত সংক্রমণ নিয়ন্ত্রণে"),
    (r"anthelmintic", "কৃমি সংক্রমণ নিরাময়ে"),
    (r"vitamin|mineral|nutritional|supplement", "পুষ্টি ও ভিটামিনের ঘাটতি পূরণে"),
    (r"steroid|corticosteroid", "তীব্র প্রদাহ ও অ্যালার্জি কমাতে"),
    (r"herbal|nutraceutical", "প্রাকৃতিক ও ভেষজ উপায়ে উপসর্গ উপশমে"),
]

SIDE_EFFECT_BN = [
    (r"nausea", "বমি ভাব"),
    (r"vomiting", "বমি"),
    (r"diarrh", "ডায়রিয়া বা পাতলা পায়খানা"),
    (r"constipation", "কোষ্ঠকাঠিন্য"),
    (r"headache", "মাথাব্যথা"),
    (r"dizziness|vertigo", "মাথা ঘোরা"),
    (r"drowsiness|sedation|somnolence", "তন্দ্রাভাব বা ঘুম ঘুম ভাব"),
    (r"dry mouth", "মুখ শুকিয়ে যাওয়া"),
    (r"rash|urticaria|pruritus|itch", "ত্বকে ফুসকুড়ি বা চুলকানি"),
    (r"stomach|abdominal|gastric|epigastric|dyspepsia", "পেটব্যথা বা গ্যাস্ট্রিক"),
    (r"fatigue|tiredness|weakness", "ক্লান্তি"),
    (r"ankle swelling|oedema|edema", "পা ফুলে যাওয়া"),
    (r"insomnia|sleep", "ঘুমের ব্যাঘাত"),
    (r"palpitation|tachycardia|heartbeat", "বুক ধড়ফড় করা"),
    (r"allergic|hypersensitivity", "অ্যালার্জির প্রতিক্রিয়া"),
]

# Separated into condition-based precautions vs standalone action warnings for clean Bangla grammar
CONDITION_PRECAUTION_BN = [
    (r"liver|hepatic", "লিভারের সমস্যা"),
    (r"kidney|renal", "কিডনির সমস্যা"),
    (r"ulcer|bleeding|gastrointestinal", "পেপটিক আলসার"),
    (r"asthma|bronchospasm", "হাঁপানি"),
    (r"hypertension|high blood pressure", "উচ্চ রক্তচাপ"),
    (r"diabetes|diabetic", "ডায়াবেটিস"),
    (r"pregnant|pregnancy|lactat|breast", "গর্ভাবস্থা বা স্তন্যদানকাল"),
]

ACTION_WARNING_BN = [
    (r"alcohol", "এই ওষুধ চলাকালীন অ্যালকোহল সম্পূর্ণ এড়িয়ে চলুন"),
    (r"driv|machin|drowsy|sedat", "সেবনের পর গাড়ি চালানো বা ঝুঁকিপূর্ণ যন্ত্রপাতি চালানো থেকে বিরত থাকুন"),
    (r"overdose|hepatotox", "অতিরিক্ত মাত্রায় সেবনে লিভারের মারাত্মক ক্ষতি হতে পারে"),
]

NO_SIDE_EFFECT_RE = re.compile(
    r"no\s+(?:known\s+|significant\s+)?side\s+effects?|generally\s+well\s+tolerated|usually\s+well\s+tolerated",
    re.IGNORECASE,
)

STRENGTH_RE = re.compile(
    r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|ml|iu)(?:\s*\+\s*\d+(?:\.\d+)?\s*(?:mg|mcg|g|ml|iu))*\b",
    re.IGNORECASE,
)

def extract_matches(text: str, rules: list, limit: int = 3) -> list:
    low = str(text or "").lower()
    found = []
    for pat, bn in rules:
        if re.search(pat, low) and bn not in found:
            found.append(bn)
            if len(found) >= limit:
                break
    return found

def extract_clean_strength(row: dict) -> str:
    """Prefer explicit dosage strength (e.g. '20 mg', '500 mg + 65 mg') when available, else package_container."""
    pack = str(row.get("package_container") or "").strip()
    dos = str(row.get("dosage_description") or "").strip()
    if STRENGTH_RE.search(pack):
        return pack
    m = STRENGTH_RE.search(dos)
    if m and len(m.group(0)) <= 24:
        return m.group(0).strip()
    return pack

def build_dosage_sentence(form_low: str, dosage_text: str, gen_low: str, cls_low: str) -> str:
    d = str(dosage_text or "").lower()
    if any(k in form_low for k in ("cream", "ointment", "gel", "lotion", "muscle rub", "scalp")):
        return "এটি খাওয়ার ওষুধ নয়; আক্রান্ত স্থানে পাতলা করে চিকিৎসকের পরামর্শমতো দিনে ১-২ বার বাহ্যিকভাবে ব্যবহার করতে হয়।"
    if any(k in form_low for k in ("ophthalmic", "eye")):
        return "চিকিৎসকের পরামর্শ অনুযায়ী আক্রান্ত চোখে নির্ধারিত ফোঁটা করে নির্দিষ্ট সময় পরপর দিতে হয়।"
    if any(k in form_low for k in ("nasal", "inhaler", "nebuliser")):
        return "চিকিৎসকের নির্দেশিত মাত্রায় ও সঠিক নিয়মে শ্বাসের সাথে বা নাকে প্রয়োগ করতে হয়।"
    if any(k in form_low for k in ("injection", "infusion")):
        return "এটি কেবল চিকিৎসক বা প্রশিক্ষিত স্বাস্থ্যকর্মীর তত্ত্বাবধানে নির্ধারিত মাত্রায় প্রয়োগ করতে হয়।"
    if "mouthwash" in form_low:
        return "চিকিৎসকের পরামর্শ অনুযায়ী নির্দিষ্ট পরিমাণ নিয়ে কুলি করতে হয়, গিলে ফেলা যাবে না।"

    is_bedtime = bool(
        re.search(r"bedtime|at night|before sleep|\bhs\b", d)
        or any(k in gen_low for k in ("diazepam", "clonazepam", "alprazolam", "bromazepam", "montelukast"))
    )
    is_before_meal = bool(
        re.search(r"before meal|before food|empty stomach|30 min|60 min", d)
        or any(k in gen_low for k in ("omeprazole", "esomeprazole", "pantoprazole", "rabeprazole", "lansoprazole", "dexlansoprazole"))
    )
    is_after_meal = bool(
        re.search(r"after meal|after food|with food|with meal", d)
        or any(k in gen_low for k in ("naproxen", "diclofenac", "ibuprofen", "tolfenamic", "aceclofenac", "ketorolac", "paracetamol", "metformin"))
    )

    if is_bedtime:
        # Never emit a contradictory 'morning & night' prefix for bedtime medicines
        return "সাধারণত রাতে ঘুমানোর আগে চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়।"

    freq = ""
    if re.search(r"4\s*-\s*6\s*hour|every 4|every 6|four times|\bqid\b", d):
        freq = "৪-৬ ঘণ্টা পরপর (দিনে ৩-৪ বার) "
    elif re.search(r"three times|thrice|\btid\b|8 hour", d):
        freq = "দিনে ৩ বার (সকাল, দুপুর ও রাত) "
    elif re.search(r"once or twice|1-2 times", d):
        freq = "দিনে ১-২ বার "
    elif re.search(r"twice|\bbid\b|12 hour|two times", d):
        freq = "দিনে ২ বার (সকাল ও রাত) "
    elif re.search(r"once daily|once a day|\bod\b|24 hour", d):
        freq = "দিনে ১ বার নির্দিষ্ট সময়ে "

    if is_before_meal:
        timing = "খাবারের ৩০ মিনিট আগে খালি পেটে"
    elif is_after_meal:
        timing = "ভরা পেটে খাবারের পরে"
    else:
        timing = "খাবারের পরে পানি দিয়ে"

    if any(k in cls_low for k in ("antibiotic", "penicillin", "cephalosporin", "macrolide", "quinolone")):
        return f"সাধারণত {freq}{timing} সেবন করতে হয় এবং চিকিৎসকের নির্ধারিত সম্পূর্ণ কোর্স শেষ করা জরুরি।"
    return f"সাধারণত {freq}{timing} চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়।"

def build_warning_sentence(row: dict, generic_en: str, cls_low: str) -> str:
    se_raw = str(row["side_effects_description"] or "").strip()
    # Respect negation: never claim side effects when the DB explicitly says 'no side effects'
    if se_raw and not NO_SIDE_EFFECT_RE.search(se_raw):
        se_list = extract_matches(se_raw, SIDE_EFFECT_BN, limit=3)
    else:
        se_list = []

    prec_raw = " ".join([
        str(row["precautions_description"]),
        str(row["contraindications_description"]),
        str(row["pregnancy_and_lactation_description"]),
        se_raw,
    ])
    cond_list = extract_matches(prec_raw, CONDITION_PRECAUTION_BN, limit=3)
    action_list = extract_matches(prec_raw, ACTION_WARNING_BN, limit=2)
    inter = str(row["interaction_description"]).strip().lower()

    parts = []
    if se_list:
        parts.append(f"এটি সেবনে {', '.join(se_list)} হতে পারে")
    if action_list:
        parts.extend(action_list)
    if cond_list:
        parts.append(f"{' বা '.join(cond_list)} থাকলে চিকিৎসকের পরামর্শ ছাড়া ব্যবহার করবেন না")
    if any(k in cls_low for k in ("antibiotic", "penicillin", "cephalosporin", "macrolide", "quinolone")):
        parts.append("সুস্থ বোধ করলেও অ্যান্টিবায়োটিকের কোর্স মাঝপথে বন্ধ করবেন না")
    elif inter and "no known" not in inter:
        parts.append("অন্য নিয়মিত ওষুধের সাথে খাওয়ার আগে চিকিৎসকের পরামর্শ নিন")

    parts.append(f"একই উপাদানের ({generic_en}) একাধিক ওষুধ একসাথে সেবন করে অতিরিক্ত মাত্রা এড়িয়ে চলুন")
    return "সতর্কতা: " + "; ".join(parts[:3]) + "।"

def build_bangla_user_prompt(row: dict, strength_str: str, question: str) -> str:
    fields = [
        ("ওষুধের নাম", row["brand_name"], 80),
        ("মাত্রা বা প্যাক", strength_str, 60),
        ("ধরন", row["dosage_form"], 50),
        ("জেনেরিক উপাদান", row["generic"], 140),
        ("প্রস্তুতকারক", row["manufacturer"], 80),
        ("ওষুধের শ্রেণি", row["drug_class"], 100),
        ("ব্যবহারের নির্দেশনা", row["indication"], 160),
        ("সেবনবিধি", row["dosage_description"], 220),
        ("পার্শ্বপ্রতিক্রিয়া", row["side_effects_description"], 160),
        ("অন্য ওষুধের সাথে বিক্রিয়া", row["interaction_description"], 150),
        ("সতর্কতা ও নিষেধাজ্ঞা", row["precautions_description"] or row["contraindications_description"], 160),
    ]
    lines = ["[ডাটাবেস তথ্য]"]
    for label, val, limit in fields:
        clean = textwrap.shorten(str(val or ""), width=limit, placeholder="...")
        if clean:
            lines.append(f"{label}: {clean}")
    lines.append(f"\nরোগীর প্রশ্ন: {question}")
    return "\n".join(lines)

def build_bangla_target(row: dict, strength_str: str, idx: int) -> str:
    # Keep brand_name, numeric strength, and generic in exact English; write all explanation in Bangla
    brand_en = str(row["brand_name"]).strip()
    generic_en = re.sub(r"\s*\[[^\]]*\]", "", str(row["generic"])).strip() or str(row["generic"]).strip()
    form_raw = str(row["dosage_form"] or "Tablet").strip()
    form_bn = FORM_BN.get(form_raw.lower(), form_raw)
    cls_low = str(row["drug_class"]).lower()
    gen_low = generic_en.lower()

    name_part = f"{brand_en} ({strength_str})" if strength_str else brand_en
    openers = [
        f"{name_part} ({form_bn}) ওষুধটির মূল জেনেরিক উপাদান হলো {generic_en}।",
        f"{name_part} হলো একটি {form_bn}, যার মূল সক্রিয় উপাদান {generic_en}।",
        f"{name_part} ({form_bn}) হলো {generic_en} উপাদানসমৃদ্ধ একটি ওষুধ।",
    ]
    s1 = openers[idx % len(openers)]

    ind_matches = extract_matches(row["indication"], INDICATION_BN_RULES, limit=3)
    if ind_matches:
        s2 = f"এটি মূলত {', '.join(ind_matches)} উপশম ও নিয়ন্ত্রণে ব্যবহৃত হয়।"
    else:
        cls_matches = extract_matches(cls_low, CLASS_BN_FALLBACK, limit=1)
        use_phrase = cls_matches[0] if cls_matches else "নির্দিষ্ট রোগের উপসর্গ উপশম ও নিয়ন্ত্রণে"
        s2 = f"এটি মূলত {use_phrase} ব্যবহৃত হয়।"

    s3 = build_dosage_sentence(form_raw.lower(), row["dosage_description"], gen_low, cls_low)
    s4 = build_warning_sentence(row, generic_en, cls_low)
    return re.sub(r"\s+", " ", f"{s1} {s2} {s3} {s4}").strip()

# Check for optional teacher-distilled dataset first
custom_candidates = [OUTPUT_DIR / "custom_bangla_sft.jsonl"] + list(Path("/kaggle/input").rglob("custom_bangla_sft.jsonl"))
custom_sft = next((p for p in custom_candidates if p.exists()), None)

TRAIN_JSONL = OUTPUT_DIR / "train.jsonl"
EVAL_JSONL = OUTPUT_DIR / "eval.jsonl"

if custom_sft is not None:
    print(f"Loading custom distilled SFT dataset: {custom_sft}")
    all_lines = [ln for ln in custom_sft.read_text(encoding="utf-8").splitlines() if ln.strip()]
    random.Random(42).shuffle(all_lines)
    eval_n = max(1, int(len(all_lines) * 0.05))
    EVAL_JSONL.write_text("\n".join(all_lines[:eval_n]) + "\n", encoding="utf-8")
    TRAIN_JSONL.write_text("\n".join(all_lines[eval_n:]) + "\n", encoding="utf-8")
    print(f"Train samples: {len(all_lines) - eval_n:,} | Eval samples: {eval_n:,}")
else:
    # Balanced sampling: up to 2 brands per unique generic across all ~1,661 generics
    balanced_df = (
        meds_df.groupby("generic_lower", group_keys=False)
        .head(2)
        .sample(frac=1.0, random_state=42)
        .reset_index(drop=True)
    )

    eval_size = max(1, int(len(balanced_df) * 0.05))
    eval_df = balanced_df.iloc[:eval_size].reset_index(drop=True)
    train_df = balanced_df.iloc[eval_size:].reset_index(drop=True)

    rng = random.Random(42)

    def write_bangla_jsonl(df, path):
        with open(path, "w", encoding="utf-8") as f:
            for idx, r in df.iterrows():
                row_dict = r.to_dict()
                strength_str = extract_clean_strength(row_dict)
                q = rng.choice(PATIENT_QUESTIONS)
                target = build_bangla_target(row_dict, strength_str, idx)
                record = {
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": build_bangla_user_prompt(row_dict, strength_str, q)},
                        {"role": "assistant", "content": target},
                    ]
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    write_bangla_jsonl(train_df, TRAIN_JSONL)
    write_bangla_jsonl(eval_df, EVAL_JSONL)
    print(f"Train samples: {len(train_df):,} -> {TRAIN_JSONL}")
    print(f"Eval samples:  {len(eval_df):,} -> {EVAL_JSONL}")

print("\n--- Sample Grounded Training Record ---")
sample = json.loads(TRAIN_JSONL.read_text(encoding="utf-8").splitlines()[0])
print(sample["messages"][1]["content"])
print("\n[ASSISTANT (English Brand/Strength/Generic + Bangla Explanation)]:\n" + sample["messages"][2]["content"])

Train samples: 2,671 -> /kaggle/working/oushudhcheck_gemma3_1b/train.jsonl
Eval samples:  140 -> /kaggle/working/oushudhcheck_gemma3_1b/eval.jsonl

--- Sample Grounded Training Record ---
[ডাটাবেস তথ্য]
ওষুধের নাম: Asmarid
মাত্রা বা প্যাক: 200 ml bottle
ধরন: Syrup
জেনেরিক উপাদান: Mahadrakkharist [Grape extract]
প্রস্তুতকারক: ACME Laboratories Ltd.
ওষুধের শ্রেণি: Herbal and Nutraceuticals
ব্যবহারের নির্দেশনা: Tuberculosis
সেবনবিধি: Mahadrakkharist content of grapes is partly responsible for the therapeutic effect. The Daily recommended dose of resveratrol is 8 mg/kg body weight. Each 5 ml syrup contains 36 mg resveratrol as a fluid extract. So...
পার্শ্বপ্রতিক্রিয়া: There are no side effects associated with the use of Drakkha in the above-mentioned therapeutic doses. Drakkha has been used safely as an ayurvedic medicine...
অন্য ওষুধের সাথে বিক্রিয়া: It should be used cautiously in patients taking anticoagulants such as warfarin, aspirin, non-steroidal anti inflammatory drugs (NSAIDS),

In [19]:
%%writefile /kaggle/working/train_ddp.py
import os
from pathlib import Path
import torch
import torch.distributed as dist
from datasets import load_dataset
from unsloth import FastModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = Path("/kaggle/working/oushudhcheck_gemma3_1b")
LORA_DIR = OUTPUT_DIR / "gemma3_1b_bangla_lora"
MODEL_ID = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"
MAX_SEQ_LEN = 1024

local_rank = int(os.environ.get("LOCAL_RANK", "0"))
world_size = int(os.environ.get("WORLD_SIZE", "1"))
torch.cuda.set_device(local_rank)

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    device_map={"": local_rank},
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

train_raw = load_dataset("json", data_files=str(OUTPUT_DIR / "train.jsonl"), split="train")
eval_raw = load_dataset("json", data_files=str(OUTPUT_DIR / "eval.jsonl"), split="train")

def apply_gemma3_template(batch):
    texts = []
    for conv in batch["messages"]:
        formatted_turns = [
            {"role": "user", "content": f"{conv[0]['content']}\n\n{conv[1]['content']}"},
            {"role": "assistant", "content": conv[2]["content"]},
        ]
        rendered = tokenizer.apply_chat_template(
            formatted_turns,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(rendered.removeprefix("<bos>"))
    return {"text": texts}

train_ds = train_raw.map(apply_gemma3_template, batched=True, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(apply_gemma3_template, batched=True, remove_columns=eval_raw.column_names)

use_bf16 = is_bfloat16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,  # Keep each medicine isolated so warnings never bleed across rows
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,  # 2 GPUs x 4 x 2 = effective batch size 16
        warmup_steps=15,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not use_bf16,
        bf16=use_bf16,
        ddp_find_unused_parameters=False,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        report_to="none",
        seed=42,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

train_stats = trainer.train()
trainer.save_model(str(LORA_DIR))

if trainer.is_world_process_zero():
    tokenizer.save_pretrained(str(LORA_DIR))
    print(f"Training finished in {train_stats.metrics['train_runtime']:.1f}s | Final loss: {train_stats.metrics['train_loss']:.4f}")
    print(f"Saved LoRA adapter -> {LORA_DIR}")

if world_size > 1 and dist.is_initialized():
    dist.barrier()
    dist.destroy_process_group()

Overwriting /kaggle/working/train_ddp.py


In [20]:
# Launch Multi-GPU DDP Training
import torch
nproc = max(1, torch.cuda.device_count())
!UNSLOTH_DEFAULT_PACKING=0 TOKENIZERS_PARALLELISM=false torchrun --nproc_per_node={nproc} /kaggle/working/train_ddp.py

W1122 10:02:43.425000 6484 torch/distributed/run.py:851] 
W1122 10:02:43.425000 6484 torch/distributed/run.py:851] *****************************************
W1122 10:02:43.425000 6484 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1122 10:02:43.425000 6484 torch/distributed/run.py:851] *****************************************
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggin

## Live Sanity & Faithfulness Audit on 5 Real Bangladeshi Medicines
Loads the trained LoRA adapter and verifies:
1. Exact English brand name, numeric strength, and generic compound are preserved (zero dose hallucinations like `65 mg -> 650 mg`).
2. Explanation, dosage form, schedule, and safety warnings are in fluent Bangla.

In [21]:
import re
import time
import torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

LORA_DIR = OUTPUT_DIR / "gemma3_1b_bangla_lora"
SYSTEM_PROMPT = (OUTPUT_DIR / "system_prompt.txt").read_text(encoding="utf-8")

model, tokenizer = FastModel.from_pretrained(
    model_name=str(LORA_DIR),
    max_seq_length=1024,
    load_in_4bit=True,
    device_map={"": 0},
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
FastModel.for_inference(model)

test_cases = [
    (
        "Napa Extra 500 mg + 65 mg",
        ["Napa Extra", "500 mg", "65 mg", "Paracetamol"],
        "[ডাটাবেস তথ্য]\nওষুধের নাম: Napa Extra\nমাত্রা বা প্যাক: 500 mg + 65 mg\nধরন: Tablet\n"
        "জেনেরিক উপাদান: Paracetamol + Caffeine\nপ্রস্তুতকারক: Beximco Pharmaceuticals Ltd.\n"
        "ব্যবহারের নির্দেশনা: Fever, headache, migraine, toothache\n"
        "সেবনবিধি: 1-2 tablets every 4-6 hours after meal\n"
        "পার্শ্বপ্রতিক্রিয়া: Nausea, allergic rash, liver damage in overdose\n\n"
        "রোগীর প্রশ্ন: এই ওষুধটি কী কাজ করে, কখন খাবো এবং কী সতর্কতা মানতে হবে?",
    ),
    (
        "Seclo 20 mg",
        ["Seclo", "20 mg", "Omeprazole"],
        "[ডাটাবেস তথ্য]\nওষুধের নাম: Seclo\nমাত্রা বা প্যাক: 20 mg\nধরন: Capsule\n"
        "জেনেরিক উপাদান: Omeprazole\nপ্রস্তুতকারক: Square Pharmaceuticals PLC\n"
        "ব্যবহারের নির্দেশনা: Gastric ulcer, acidity, GERD, heartburn\n"
        "সেবনবিধি: 20 mg once or twice daily 30 minutes before meal\n"
        "পার্শ্বপ্রতিক্রিয়া: Headache, nausea, diarrhea\n\n"
        "রোগীর প্রশ্ন: এই ওষুধটা কি খাবারের আগে না পরে খেতে হয়? এর কাজ ও সতর্কতা বলুন।",
    ),
    (
        "Amdocal 5 mg",
        ["Amdocal", "5 mg", "Amlodipine"],
        "[ডাটাবেস তথ্য]\nওষুধের নাম: Amdocal\nমাত্রা বা প্যাক: 5 mg\nধরন: Tablet\n"
        "জেনেরিক উপাদান: Amlodipine Besilate\nপ্রস্তুতকারক: Beximco Pharmaceuticals Ltd.\n"
        "ব্যবহারের নির্দেশনা: Hypertension (High blood pressure), Angina\n"
        "সেবনবিধি: 5 mg once daily\n"
        "পার্শ্বপ্রতিক্রিয়া: Dizziness, headache, ankle swelling, fatigue\n\n"
        "রোগীর প্রশ্ন: সহজ বাংলায় এই ওষুধটির কাজ, খাওয়ার নিয়ম ও সতর্কতা দিন।",
    ),
    (
        "Tufnil 200 mg",
        ["Tufnil", "200 mg", "Tolfenamic"],
        "[ডাটাবেস তথ্য]\nওষুধের নাম: Tufnil\nমাত্রা বা প্যাক: 200 mg\nধরন: Tablet\n"
        "জেনেরিক উপাদান: Tolfenamic Acid\nপ্রস্তুতকারক: Eskayef Pharmaceuticals Ltd.\n"
        "ব্যবহারের নির্দেশনা: Acute migraine headache, pain\n"
        "সেবনবিধি: 200 mg after meal at onset of migraine\n"
        "পার্শ্বপ্রতিক্রিয়া: Stomach discomfort, nausea, diarrhea\n"
        "সতর্কতা ও নিষেধাজ্ঞা: Peptic ulcer, asthma, kidney disease\n\n"
        "রোগীর প্রশ্ন: মাইগ্রেনের এই ওষুধটি কীভাবে খাবো এবং কী সাবধানতা মানতে হবে?",
    ),
    (
        "Diazepam 5 mg",
        ["Diazepam", "5 mg"],
        "[ডাটাবেস তথ্য]\nওষুধের নাম: Diazepam\nমাত্রা বা প্যাক: 5 mg\nধরন: Tablet\n"
        "জেনেরিক উপাদান: Diazepam\nওষুধের শ্রেণি: Benzodiazepine\n"
        "ব্যবহারের নির্দেশনা: Anxiety, insomnia, muscle spasm, seizures\n"
        "সেবনবিধি: 5 mg at bedtime as prescribed\n"
        "পার্শ্বপ্রতিক্রিয়া: Drowsiness, dizziness, fatigue\n"
        "সতর্কতা ও নিষেধাজ্ঞা: Avoid alcohol and driving\n\n"
        "রোগীর প্রশ্ন: বয়স্ক রোগীর জন্য এই ওষুধ খাওয়ার নিয়ম ও জরুরি সতর্কতা সহজ বাংলায় লিখুন।",
    ),
]

for name, required_tokens, user_text in test_cases:
    messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_text}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda:0")

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.15,
            top_p=0.9,
            repetition_penalty=1.05,
            do_sample=True,
        )
    elapsed_ms = int((time.perf_counter() - t0) * 1000)
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    missing = [tok for tok in required_tokens if tok.lower() not in generated.lower()]
    has_bangla = bool(re.search(r"[\u0980-\u09FF]{15,}", generated))
    status = "PASS" if (not missing and has_bangla) else f"CHECK (missing={missing}, bangla={has_bangla})"

    print(f"=== {name} | {elapsed_ms} ms | {status} ===")
    print(generated)
    print()

==((====))==  Unsloth 2026.11.9: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Unsloth: the model has bfloat16 layers on GPU 0 (Tesla T4), which has no native bfloat16, so they run slowly. Load with dtype = torch.float16, or keep the model on bfloat16 GPUs with CUDA_VISIBLE_DEVICES.


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Napa Extra 500 mg + 65 mg | 2895 ms | CHECK (missing=[], bangla=False) ===
Napa Extra (500 mg + 65 mg) হলো একটি ট্যাবলেট, যার মূল সক্রিয় উপাদান Paracetamol + Caffeine। এটি মূলত জ্বর, মাথাব্যথা, দাঁতব্যথা উপশম ও নিয়ন্ত্রণে ব্যবহৃত হয়। সাধারণত ৪-৬ ঘণ্টা পরপর (দিনে ৩-৪ বার) খাবারের পরে পানি দিয়ে চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়। সতর্কতা: এটি সেবনে বমি ভাব, ত্বকে ফুসকুড়ি বা চুলকানি হতে পারে; লিভারের সমস্যা বা গর্ভাবস্থা বা স্তন্যদানকাল থাকলে চিকিৎসকের পরামর্শ ছাড়া ব্যবহার করবেন না; একই উপাদানের (Paracetamol + Caffeine) একাধিক ওষুধ একসাথে সেবন করে অতিরিক্ত মাত্রা এড়িয়ে চলুন।



Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Seclo 20 mg | 2387 ms | CHECK (missing=[], bangla=False) ===
Seclo (20 mg) (ক্যাপসুল) হলো Omeprazole উপাদানসমৃদ্ধ একটি ওষুধ। এটি মূলত গ্যাস্ট্রিক, অ্যাসিডিটি ও বুকজ্বালা উপশম ও নিয়ন্ত্রণে ব্যবহৃত হয়। সাধারণত দিনে ১-২ বার খাবারের পরে পানি দিয়ে চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়। সতর্কতা: এটি সেবনে বমি ভাব, ডায়রিয়া বা পাতলা পায়খানা, মাথাব্যথা হতে পারে; পেপটিক আলসার বা গর্ভাবস্থা বা স্তন্যদানকাল থাকলে চিকিৎসকের পরামর্শ ছাড়া ব্যবহার করবেন না; একই উপাদানের (Omeprazole) একাধিক ওষুধ একসাথে সেবন করে অতিরিক্ত মাত্রা এড়িয়ে চলুন।



Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Amdocal 5 mg | 2144 ms | CHECK (missing=[], bangla=False) ===
Amdocal (5 mg) হলো একটি ট্যাবলেট, যার মূল সক্রিয় উপাদান Amlodipine Besilate। এটি মূলত উচ্চ রক্তচাপ, অতিরিক্ত উদ্বেগ ও অস্থিরতা উপশম ও নিয়ন্ত্রণে ব্যবহৃত হয়। সাধারণত দিনে ১ বার নির্দিষ্ট সময়ে খাবারের পরে পানি দিয়ে চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়। সতর্কতা: এটি সেবনে মাথাব্যথা হতে পারে; কিডনির সমস্যা বা গর্ভাবস্থা বা স্তন্যদানকাল থাকলে চিকিৎসকের পরামর্শ ছাড়া ব্যবহার করবেন না; একই উপাদানের (Amlodipine Besilate) একাধিক ওষুধ একসাথে সেবন করে অতিরিক্ত মাত্রা এড়িয়ে চলুন।



Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Tufnil 200 mg | 2692 ms | CHECK (missing=[], bangla=False) ===
Tufnil (200 mg) (ট্যাবলেট) হলো Tolfenamic Acid উপাদানসমৃদ্ধ একটি ওষুধ। এটি মূলত মাথাব্যথা উপশম ও নিয়ন্ত্রণে ব্যবহৃত হয়। সাধারণত খাবারের পরে পানি দিয়ে চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়। সতর্কতা: এটি সেবনে বমি ভাব, ডায়রিয়া বা পাতলা পায়খানা, পেটব্যথা বা গ্যাস্ট্রিক হতে পারে; কিডনির সমস্যা বা পেপটিক আলসার বা গর্ভাবস্থা বা স্তন্যদানকাল থাকলে চিকিৎসকের পরামর্শ ছাড়া ব্যবহার করবেন না; একই উপাদানের (Tolfenamic Acid) একাধিক ওষুধ একসাথে সেবন করে অতিরিক্ত মাত্রা এড়িয়ে চলুন।

=== Diazepam 5 mg | 1569 ms | CHECK (missing=[], bangla=False) ===
Diazepam (5 mg) হলো একটি ট্যাবলেট, যার মূল সক্রিয় উপাদান Diazepam। এটি মূলত উদ্বেগ, অনিদ্রা ও স্নায়বিক চাপ কমাতে ব্যবহৃত হয়। সাধারণত রাতে ঘুমানোর আগে চিকিৎসকের নির্ধারিত মাত্রা অনুযায়ী সেবন করতে হয়। সতর্কতা: এই ওষুধ চলাকালীন অ্যালকোহল সম্পূর্ণ এড়িয়ে চলুন; একই উপাদানের (Diazepam) একাধিক ওষুধ একসাথে সেবন করে অতিরিক্ত মাত্রা এড়িয়ে চলুন।



## Export Quantized `Q4_K_M` GGUF + Ollama `Modelfile`
Merges the LoRA weights, exports a `Q4_K_M` GGUF, copies it right next to `Modelfile` in `/kaggle/working/oushudhcheck_gemma3_1b/`, and prints the exact verification command.

In [23]:
import shutil

GGUF_DIR = OUTPUT_DIR / "gemma3_1b_bangla_gguf"
FINAL_GGUF_NAME = "oushudhcheck-gemma3-1b-bn.Q4_K_M.gguf"
FINAL_GGUF_PATH = OUTPUT_DIR / FINAL_GGUF_NAME
SYSTEM_PROMPT = (OUTPUT_DIR / "system_prompt.txt").read_text(encoding="utf-8")

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_candidates = sorted(
    set(OUTPUT_DIR.rglob("*.gguf")),
    key=lambda p: p.stat().st_size,
)
assert gguf_candidates, f"No .gguf file found anywhere under {OUTPUT_DIR}!"

# Pick the quantized q4_k_m file (smallest valid GGUF, not the ~2GB f16 intermediate)
q4_file = next((p for p in gguf_candidates if "q4_k_m" in p.name.lower()), gguf_candidates[0])
if q4_file.resolve() != FINAL_GGUF_PATH.resolve():
    shutil.copy2(q4_file, FINAL_GGUF_PATH)

print(f"Ready-to-download GGUF -> {FINAL_GGUF_PATH} ({FINAL_GGUF_PATH.stat().st_size / (1024 * 1024):.1f} MB)")

# Modelfile sits in the exact same directory as FINAL_GGUF_NAME so `ollama create` works directly
modelfile_path = OUTPUT_DIR / "Modelfile"
modelfile_path.write_text(
    f"FROM ./{FINAL_GGUF_NAME}\n"
    "PARAMETER temperature 0.15\n"
    "PARAMETER top_p 0.9\n"
    "PARAMETER repeat_penalty 1.05\n"
    "PARAMETER num_ctx 1024\n"
    f'SYSTEM """{SYSTEM_PROMPT}"""\n',
    encoding="utf-8",
)
print("Saved Ollama Modelfile ->", modelfile_path)
print("\nDownload these 2 files from Kaggle Output (/kaggle/working/oushudhcheck_gemma3_1b/):")
print(f"  1. {FINAL_GGUF_NAME}")
print("  2. Modelfile")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.26s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/oushudhcheck_gemma3_1b/gemma3_1b_bangla_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/kaggle/working/oushudhcheck_gemma3_1b/gemma3_1b_bangla_gguf_gguf/gemma-3-1b-it.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/kaggle/working/oushudhcheck_gemma3_1b/gemma3_1b_bangla_gguf_gguf/gemma-3-1b-it.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /kaggle/working/oushudhcheck_gemma3_1b/gemma3_1b_bangla_gguf_gguf/gemma-3-1b-it.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /kaggle/working/oushudhcheck_gemma3_1b/gemma3_1b_bangla_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /kaggle/working/oushudhcheck_gemma3_1b/gemma3_1b_bangla_gguf_gguf/Modelfile
Ready-to-download GGUF -> /kaggle/working/oushudhcheck_gemma3_1b/oushudhcheck-gemma3-1b-bn.Q4_K_M.gguf (768.7 MB)
Saved Ollama Modelfile -> /kaggle/working/oushudhcheck_gemma3_1b/Modelfile

Download these 2 files from Kaggle Output (/kaggle/working/oushudhcheck_gemma3_1b/):
  1. oushudhcheck-gemma3-1b-bn.Q4_K_M.gguf
  2. Modelf

In [1]:
# EXPORTING ZIP FILE

import zipfile                                                                                                        
from pathlib import Path                                                                                              
from IPython.display import FileLink, display                                                                         
                                                                                                                      
OUTPUT_DIR = Path("/kaggle/working/oushudhcheck_gemma3_1b")                                                           
ZIP_PATH = Path("/kaggle/working/oushudhcheck_gemma3_1b_deploy.zip")                                                  
                                                                                                                      
files_to_pack = [                                                                                                     
    OUTPUT_DIR / "oushudhcheck-gemma3-1b-bn.Q4_K_M.gguf",                                                             
    OUTPUT_DIR / "Modelfile",                                                                                         
    OUTPUT_DIR / "system_prompt.txt",                                                                                 
]                                                                                                                     
                                                                                                                      
lora_dir = OUTPUT_DIR / "gemma3_1b_bangla_lora"                                                                       
                                                                                                                      
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_STORED) as zf:                                            
    for f in files_to_pack:                                                                                           
        if f.exists():                                                                                                
            zf.write(f, arcname=f.name)                                                                               
            print(f"Added: {f.name} ({f.stat().st_size / (1024 * 1024):.1f} MB)")                                     
        else:                                                                                                         
            print(f"Missing (skipped): {f}")                                                                          
                                                                                                                      
    if lora_dir.exists():                                                                                             
        for lf in sorted(lora_dir.rglob("*")):                                                                        
            if lf.is_file() and ".unsloth_tp_rank" not in str(lf):                                                    
                zf.write(lf, arcname=f"gemma3_1b_bangla_lora/{lf.relative_to(lora_dir)}")                             
        print("Added: gemma3_1b_bangla_lora/")                                                                        
                                                                                                                      
print(f"\nZIP Ready -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024 * 1024):.1f} MB)")                                
display(FileLink(str(ZIP_PATH.relative_to("/kaggle/working"))))    

Missing (skipped): /kaggle/working/oushudhcheck_gemma3_1b/oushudhcheck-gemma3-1b-bn.Q4_K_M.gguf
Missing (skipped): /kaggle/working/oushudhcheck_gemma3_1b/Modelfile
Missing (skipped): /kaggle/working/oushudhcheck_gemma3_1b/system_prompt.txt

ZIP Ready -> /kaggle/working/oushudhcheck_gemma3_1b_deploy.zip (0.0 MB)


/kaggle/working/oushudhcheck_gemma3_1b_deploy.zip